<a href="https://colab.research.google.com/github/AhmadObaidat/School/blob/main/AIT509_Wk_2_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sentiment_dataset.csv to sentiment_dataset (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv("sentiment_dataset.csv")
df.head()

,text,label
0,I love this product,positive
1,This phone is amazing,positive
2,The service was excellent,positive
3,Very happy with my purchase,positive
4,The quality is great,positive


In [ ]:
print("Dataset shape:", df.shape)
print(df["label"].value_counts())

Dataset shape: (60, 2)
label
positive    30
negative    30
Name: count, dtype: int64


In [ ]:
import re

positive_words = {
    "love", "amazing", "excellent", "happy", "great", "perfectly", "satisfied",
    "wonderful", "beautiful", "best", "clear", "rich", "easy", "quick",
    "exceeded", "smoothly", "recommend", "strong", "nice", "sharp", "bright",
    "good", "premium", "reliable", "impressed", "helpful", "fits", "responsive"
}

negative_words = {
    "terrible", "hate", "disappointed", "awful", "waste", "broke", "bad",
    "regret", "poor", "damaged", "dies", "unhelpful", "cheap", "crashes",
    "stopped", "dull", "blurry", "confusing", "frustrating", "flimsy",
    "late", "broken", "hard", "unclear", "weak", "awkward", "uncomfortable",
    "freezes", "inconsistent"
}

negation_words = {"not", "no", "never", "don't", "didn't", "isn't", "wasn't", "wouldn't"}

def tokenize(text):
    return re.findall(r"\b\w+(?:'\w+)?\b", text.lower())

def rule_based_sentiment(text):
    tokens = tokenize(text)
    score = 0

    for i, token in enumerate(tokens):
        prev_token = tokens[i - 1] if i > 0 else ""
        negated = prev_token in negation_words

        if token in positive_words:
            score += -1 if negated else 1
        elif token in negative_words:
            score += 1 if negated else -1

    if score > 0:
        return "positive"
    else:
        return "negative"

df["rule_prediction"] = df["text"].apply(rule_based_sentiment)
df[["text", "label", "rule_prediction"]].head(10)

,text,label,rule_prediction
0,I love this product,positive,positive
1,This phone is amazing,positive,positive
2,The service was excellent,positive,positive
3,Very happy with my purchase,positive,positive
4,The quality is great,positive,positive
5,Fast shipping and great packaging,positive,positive
6,This item works perfectly,positive,positive
7,I am satisfied with the results,positive,positive
8,Absolutely wonderful experience,positive,positive
9,The design is beautiful,positive,positive


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rule_accuracy = accuracy_score(df["label"], df["rule_prediction"])
print("Rule-Based Accuracy:", rule_accuracy)
print("\nClassification Report:\n")
print(classification_report(df["label"], df["rule_prediction"]))
print("Confusion Matrix:\n", confusion_matrix(df["label"], df["rule_prediction"]))

Rule-Based Accuracy: 0.95

Classification Report:

              precision    recall  f1-score   support

    negative       0.91      1.00      0.95        30
    positive       1.00      0.90      0.95        30

    accuracy                           0.95        60
   macro avg       0.95      0.95      0.95        60
weighted avg       0.95      0.95      0.95        60

Confusion Matrix:
 [[30  0]
 [ 3 27]]


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

vectorizer = CountVectorizer(lowercase=True, stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

nb_predictions = nb_model.predict(X_test_vec)

In [ ]:
nb_accuracy = accuracy_score(y_test, nb_predictions)
print("Naive Bayes Accuracy:", nb_accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, nb_predictions))
print("Confusion Matrix:\n", confusion_matrix(y_test, nb_predictions))

Naive Bayes Accuracy: 0.4444444444444444

Classification Report:

              precision    recall  f1-score   support

    negative       0.47      0.89      0.62         9
    positive       0.00      0.00      0.00         9

    accuracy                           0.44        18
   macro avg       0.24      0.44      0.31        18
weighted avg       0.24      0.44      0.31        18

Confusion Matrix:
 [[8 1]
 [9 0]]


In [ ]:
sample_texts = [
    "I love the design and the quality is great",
    "This product is terrible and broke quickly",
    "The product is not bad",
    "I would not recommend this item",
    "The service was okay but the packaging was damaged",
    "Setup was easy but performance is inconsistent"
]

sample_df = pd.DataFrame({"text": sample_texts})
sample_df["rule_prediction"] = sample_df["text"].apply(rule_based_sentiment)
sample_df["nb_prediction"] = nb_model.predict(vectorizer.transform(sample_df["text"]))

sample_df

,text,rule_prediction,nb_prediction
0,I love the design and the quality is great,positive,positive
1,This product is terrible and broke quickly,negative,negative
2,The product is not bad,positive,negative
3,I would not recommend this item,negative,negative
4,The service was okay but the packaging was dam...,negative,negative
5,Setup was easy but performance is inconsistent,negative,negative


In [ ]:
df.to_csv("full_results_with_rule_predictions.csv", index=False)
sample_df.to_csv("sample_comparison_results.csv", index=False)

from google.colab import files
files.download("full_results_with_rule_predictions.csv")
files.download("sample_comparison_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
errors = df[df["label"] != df["rule_prediction"]]

print("Rule-based errors:")
errors

Rule-based errors:


,text,label,rule_prediction
29,Everything arrived on time,positive,negative
31,I really like how simple it is to use,positive,negative
39,It does exactly what I needed,positive,negative


In [ ]:
comparison = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "nb_prediction": nb_predictions
})

nb_errors = comparison[comparison["true_label"] != comparison["nb_prediction"]]

print("Naive Bayes errors:")
nb_errors

Naive Bayes errors:


,text,true_label,nb_prediction
25,This exceeded my expectations,positive,negative
28,Strong build quality and nice finish,positive,negative
40,The battery dies too quickly,negative,positive
9,The design is beautiful,positive,negative
24,Setup was easy and quick,positive,negative
35,The instructions were clear and helpful,positive,negative
29,Everything arrived on time,positive,negative
21,Customer support was very helpful,positive,negative
38,The controls are responsive,positive,negative
37,I had a good experience overall,positive,negative
